Initial setup:
```
source /global/cfs/cdirs/desi/software/desi_environment.sh 21.2
module load parallel

export INDIR=/global/cfs/cdirs/desi/spectro/redux/cascades/exposures
export OUTDIR=/global/cfs/cdirs/desi/users/$USER/redux/cascades/single_exposures
```

Run desi_group_spectra:
```
salloc -N 1 -C haswell -q interactive -t 4:00:00
parallel --jobs 32 < desi_group_spectra.txt ; exit
```

Run desi_coadd_spectra:
```
salloc -N 1 -C haswell -q interactive -t 4:00:00
parallel --jobs 32 < desi_coadd_spectra.txt ; exit
```

Run redrock:
```
salloc -N 40 --qos interactive -C haswell -t 4:00:00
parallel --jobs 40 --delay 1 < rrdesi.txt ; exit
```

In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
n_exp = 1
overwrite = False

# indir = os.getenv('INDIR')
# output_dir = os.getenv('OUTDIR')
indir = '/global/cfs/cdirs/desi/spectro/redux/cascades/exposures'
outdir = os.path.expandvars('/global/cfs/cdirs/desi/users/$USER/redux/cascades/single_exposures')
os.environ['INDIR'] = indir
os.environ['OUTDIR'] = outdir

explist = Table.read('/global/cfs/cdirs/desi/spectro/redux/cascades/run/scripts/tiles/explist-all-sv1.txt', format='ascii')
print(len(explist))

1141


In [4]:
explist[:1]

TILEID,NIGHT,EXPID
int64,int64,int64
80238,20201208,66597


In [5]:
cframes_stack = []

for index in range(len(explist)):
    tileid = explist['TILEID'][index]
    night = explist['NIGHT'][index]
    expid = explist['EXPID'][index]
    exposure_dir = os.path.join(indir, str(night), str(expid).zfill(8))
    cframe_list = glob.glob(os.path.join(exposure_dir, 'cframe-*'))
    if len(cframe_list)==0:
        print(exposure_dir, 'is empty')
        continue

    cframes = Table()
    cframes['cframe_fn'] = np.array(cframe_list)
    cframes['tileid'] = np.zeros(len(cframes), dtype=int)
    cframes['expid'] = np.zeros(len(cframes), dtype=int)
    cframes['camera'] = ' '
    cframes['petal_loc'] = -1 * np.ones(len(cframes), dtype=np.int32)

    for cframe_index, cframe_fn in enumerate(cframe_list):
        _, cp, expid1 = os.path.basename(cframe_fn).strip('.fits').split('-')
        camera, petal_loc = cp[0], cp[1]
        expid1 = int(expid)
        if expid1!=expid:
            raise ValueError()
        cframes['tileid'][cframe_index] = tileid
        cframes['expid'][cframe_index] = expid
        cframes['camera'][cframe_index] = camera
        cframes['petal_loc'][cframe_index] = petal_loc
    
    cframes_stack.append(cframes)
    
cframes = vstack(cframes_stack)
print(len(cframes))
print(len(np.unique(cframes['expid'])))

cframes[:1]

/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201208/00066597 is empty
/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201208/00066601 is empty
/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201208/00066604 is empty
/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201208/00066606 is empty
/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201208/00066608 is empty
/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201209/00066641 is empty
31281
1135


cframe_fn,tileid,expid,camera,petal_loc
str97,int64,int64,str1,int32
/global/cfs/cdirs/desi/spectro/redux/cascades/exposures/20201214/00067710/cframe-r0-00067710.fits,80605,67710,r,0


In [6]:
cframes['keep'] = False
for expid in np.unique(cframes['expid']):
    mask = cframes['expid']==expid
    for petal_loc in np.arange(10):
        mask1 = mask & (cframes['petal_loc']==petal_loc)
        if np.sum(mask1)==3:  # require all 3 cameras
            cframes['keep'][mask1] = True
print(np.sum(cframes['keep']), np.sum(~cframes['keep']))

cframes = cframes[cframes['keep']]
print(len(cframes))

31269 12
31269


In [7]:
# mask = cframes['tileid']==65008
# cframes = cframes[mask]

In [8]:
# keep only one camera for simplicity
mask = cframes['camera']=='b'
cframes = cframes[mask]
print(len(cframes))

cframes.sort('cframe_fn')

10423


## Print desi_group_spectra commands

In [9]:
command_output_path_all = 'desi_group_spectra_all.txt'
command_output_path = 'desi_group_spectra.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for expid in np.unique(cframes['expid']):

    mask = (cframes['expid']==expid)
    tileid = cframes['tileid'][mask][0]
    
    for petal_loc in np.unique(cframes['petal_loc'][mask]):
        
        exposure_dir = os.path.dirname(cframes['cframe_fn'][mask][0])
        input_loc = os.path.join(exposure_dir.replace(indir, '$INDIR'), 'cframe-[brz]{}-*.fits'.format(petal_loc))

        # Stephen's suggested directory and filename format
        expid_str = str(expid).zfill(8)
        spectra_output_loc = os.path.join('$OUTDIR', str(tileid), 'spectra-{}-{}-{}.fits'.format(petal_loc, tileid, expid_str))

        f_all.write('desi_group_spectra --inframes {} --outfile {}\n'.format(input_loc, spectra_output_loc))
        # print('desi_group_spectra --inframes {} --outfile {}'.format(input_argument, output_argument))
        
        if os.path.isfile(os.path.expandvars(spectra_output_loc)) and (not overwrite):
            # print('Warninig: {} already exists!'.format(spectra_output_loc))
            pass
        else:
            f.write('desi_group_spectra --inframes {} --outfile {}\n'.format(input_loc, spectra_output_loc))

f.close()
f_all.close()

## Print coadd_group_spectra commands

In [10]:
command_output_path_all = 'desi_coadd_spectra_all.txt'
command_output_path = 'desi_coadd_spectra.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for expid in np.unique(cframes['expid']):

    mask = (cframes['expid']==expid)
    tileid = cframes['tileid'][mask][0]

    for petal_loc in np.unique(cframes['petal_loc'][mask]):

        exposure_dir = os.path.dirname(cframes['cframe_fn'][mask][0])
        input_loc = os.path.join(exposure_dir.replace(indir, '$INDIR'), 'cframe-[brz]{}-*.fits'.format(petal_loc))

        # Stephen's suggested directory and filename format
        expid_str = str(expid).zfill(8)
        coadd_output_loc = os.path.join('$OUTDIR', str(tileid), 'coadd-{}-{}-{}.fits'.format(petal_loc, tileid, expid_str))

        f_all.write('desi_coadd_spectra -i {} -o {}\n'.format(input_loc, coadd_output_loc))
        # print('desi_coadd_spectra -i {} -o {}'.format(input_argument, output_argument))

        if os.path.isfile(os.path.expandvars(coadd_output_loc)) and (not overwrite):
            # print('Warninig: {} already exists!'.format(coadd_output_loc))
            pass
        else:
            f.write('desi_coadd_spectra -i {} -o {}\n'.format(input_loc, coadd_output_loc))

f.close()
f_all.close()

## Print redrock commands

Include 10-minute timeout

In [11]:
command_output_path_all = 'rrdesi_all.txt'
command_output_path = 'rrdesi.txt'  # commands for reruns (e.g. after failures)

f_all = open(command_output_path_all, 'w')
f = open(command_output_path, 'w')

for expid in np.unique(cframes['expid']):

    mask = (cframes['expid']==expid)
    tileid = cframes['tileid'][mask][0]
    
    for petal_loc in np.unique(cframes['petal_loc'][mask]):
        
        # Stephen's suggested directory and filename format
        expid_str = str(expid).zfill(8)
        spectra_output_loc = os.path.join('$OUTDIR', str(tileid), 'spectra-{}-{}-{}.fits'.format(petal_loc, tileid, expid_str))

        redrock_output_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-{}.h5'.format(petal_loc, tileid, expid_str))
        zbest_output_loc = os.path.join('$OUTDIR', str(tileid), 'zbest-{}-{}-{}.fits'.format(petal_loc, tileid, expid_str))
        log_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-{}.log'.format(petal_loc, tileid, expid_str))

        f_all.write('srun -N 1 -n 32 -c 2 -t 00:10:00 rrdesi_mpi {} -o {} -z {} &> {}\n'.format(spectra_output_loc, redrock_output_loc, zbest_output_loc, log_loc))
        
        if os.path.isfile(os.path.expandvars(zbest_output_loc)) and (not overwrite):
            # print('Warninig: {} already exists!'.format(zbest_output_loc))
            pass
        else:
            f.write('srun -N 1 -n 32 -c 2 -t 00:10:00 rrdesi_mpi {} -o {} -z {} &> {}\n'.format(spectra_output_loc, redrock_output_loc, zbest_output_loc, log_loc))
f.close()
f_all.close()

__Split the jobs into six batches for 40-node runs__

In [12]:
cframes_all = cframes.copy()

In [14]:
idx_split = np.array_split(np.arange(len(cframes)), 6)
for batch_index in range(len(idx_split)):
    command_output_path = 'rrdesi_{}.txt'.format(batch_index+1)  # commands for reruns (e.g. after failures)
    cframes = cframes_all[idx_split[batch_index]]
    
    print(len(cframes))

    f = open(command_output_path, 'w')

    for expid in np.unique(cframes['expid']):

        mask = (cframes['expid']==expid)
        tileid = cframes['tileid'][mask][0]

        for petal_loc in np.unique(cframes['petal_loc'][mask]):

            # Stephen's suggested directory and filename format
            expid_str = str(expid).zfill(8)
            spectra_output_loc = os.path.join('$OUTDIR', str(tileid), 'spectra-{}-{}-{}.fits'.format(petal_loc, tileid, expid_str))

            redrock_output_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-{}.h5'.format(petal_loc, tileid, expid_str))
            zbest_output_loc = os.path.join('$OUTDIR', str(tileid), 'zbest-{}-{}-{}.fits'.format(petal_loc, tileid, expid_str))
            log_loc = os.path.join('$OUTDIR', str(tileid), 'redrock-{}-{}-{}.log'.format(petal_loc, tileid, expid_str))

            if os.path.isfile(os.path.expandvars(zbest_output_loc)) and (not overwrite):
                # print('Warninig: {} already exists!'.format(zbest_output_loc))
                pass
            else:
                f.write('srun -N 1 -n 32 -c 2 -t 00:10:00 rrdesi_mpi {} -o {} -z {} &> {}\n'.format(spectra_output_loc, redrock_output_loc, zbest_output_loc, log_loc))
    
    f.close()

1738
1737
1737
1737
1737
1737
